### Loading Datasets

In [231]:
import pandas as pd

df = pd.read_csv("/content/machine_failure_dataset.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Temperature   1000 non-null   float64
 1   Vibration     1000 non-null   float64
 2   Power_Usage   1000 non-null   float64
 3   Humidity      1000 non-null   float64
 4   Machine_Type  1000 non-null   object 
 5   Failure_Risk  1000 non-null   int64  
dtypes: float64(4), int64(1), object(1)
memory usage: 47.0+ KB


There is a 0 null because all is consistent at 1000 data<br>
There is a one categorical columns with name "Machine_Type"<br>
And the target is __Failure_Risk__ i know from read documentation.

__Lets see the preview of datasets__

In [232]:
df.head()

,Temperature,Vibration,Power_Usage,Humidity,Machine_Type,Failure_Risk
0,74.967142,56.996777,8.649643,20.460962,Mill,1
1,68.617357,54.623168,9.710963,25.698075,Lathe,0
2,76.476885,50.298152,8.415160,27.931972,Drill,1
3,85.230299,46.765316,9.384077,39.438438,Lathe,1
4,67.658466,53.491117,6.212771,32.782766,Drill,1


__Lets see the description of datasets__

In [233]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Temperature,1000.0,70.193321,9.792159,37.587327,63.524097,70.253006,76.479439,108.527315
Vibration,1000.0,50.354181,4.987272,35.298057,46.968792,50.315386,53.644411,65.965538
Power_Usage,1000.0,10.011668,1.966909,3.960976,8.704001,9.999498,11.321831,17.852475
Humidity,1000.0,29.906404,5.135663,15.352757,26.312898,30.000923,33.334727,46.215465
Failure_Risk,1000.0,0.300000,0.458487,0.000000,0.000000,0.000000,1.000000,1.000000


From numerical column description i see the min and max is suitable and no outlier data

__Lets see descrition of categorical columns__

In [234]:
df.describe(include="object").T

,count,unique,top,freq
Machine_Type,1000,3,Lathe,338


Okey thats normal categorical columns

Lets see the uniques categorical column

In [235]:
df["Machine_Type"].unique()

array(['Mill', 'Lathe', 'Drill'], dtype=object)

All right that normal, there is no one anomalies values, so we no needed to clean it!

__Makesure again to check duplicate and miss values__

In [236]:
print(f"Duplicated value: {df.duplicated().sum()}")

df.isna().sum().rename("Missing values")

Duplicated value: 0


,Missing values
Temperature,0
Vibration,0
Power_Usage,0
Humidity,0
Machine_Type,0
Failure_Risk,0


Yes there is no zero duplicate and missing values. So we no needed to clean it!

__Change Cateogrical columns to numerical__

In [237]:
cat_to_num = {
    'Mill': 0,
    'Lathe': 1,
    'Drill': 2
}

df["Machine_Type"].replace(cat_to_num, inplace=True)

# Lets see the changes
df["Machine_Type"].head()

<ipython-input-237-16e8a2ceba30>:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Machine_Type"].replace(cat_to_num, inplace=True)
<ipython-input-237-16e8a2ceba30>:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Machine_Type"].replace(cat_to_num, inplace=True)


,Machine_Type
0,0
1,1
2,2
3,1
4,2


Alright that works, now all datasets is clear now we will split that data

### Splitting Data

In [239]:
from sklearn.model_selection import train_test_split

# Select the train feature (X) and target (y)
X = df.drop(columns="Failure_Risk")
y = df.Failure_Risk

# Split to train and test data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Normalize Data
I normalize data with Standart Scaler for easy model learning soon

In [240]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Modeling<br>
I choose LogisticRegression because that model is suitable for our datasets: simple, light, and easy to use

In [241]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier()
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

### Evaluation<br>
Lets evaluation DecisionTreeClassifier model<br>
Here evaluation technique i use:
+ Accuracy<br>
$\frac{TP + TN}{TP + TN + FP + FN}$
+ Precision<br>
$\frac{TP}{TP + FP}$
+ Recal<br>
$\frac{TP}{TP + FN}$
+ F1-Score<br>
$\frac{2 \cdot \text{Precision} \cdot \text{Recal}}{\text{Precision} + \text{Recal}}$

In [242]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

acc_score = accuracy_score(y_test, y_pred)
prec_score = precision_score(y_test, y_pred)
rec_score = recall_score(y_test, y_pred)
f1_score = 2 * (prec_score * rec_score) / (prec_score + rec_score)

print(f"Accuracy Score: {acc_score}")
print(f"Precision Score: {prec_score}")
print(f"Recall Score: {rec_score}")
print(f"F1 Score: {f1_score}")

Accuracy Score: 0.58
Precision Score: 0.3492063492063492
Recall Score: 0.3384615384615385
F1 Score: 0.34375


__Oke the result is:__<br>
Accuracy Score  : 0.58<br>
Precision Score : 0.34<br>
Recall Score    : 0.33<br>
F1 Score        : 0.34<br>

The results is bad because the sample of datasets is also bad, let we see the correlation

In [243]:
df.corr()

,Temperature,Vibration,Power_Usage,Humidity,Machine_Type,Failure_Risk
Temperature,1.000000,-0.040400,0.022129,-0.013321,0.000343,0.029938
Vibration,-0.040400,1.000000,-0.011199,-0.054698,-0.049877,-0.001727
Power_Usage,0.022129,-0.011199,1.000000,0.021586,0.046435,0.020971
Humidity,-0.013321,-0.054698,0.021586,1.000000,0.001561,-0.018007
Machine_Type,0.000343,-0.049877,0.046435,0.001561,1.000000,0.030039
Failure_Risk,0.029938,-0.001727,0.020971,-0.018007,0.030039,1.000000


As we know averagge all of data is not approach at 1.0 or -1.0